# Week 7 — Gradient Boosting I: XGBoost + SHAP
**Date:** July 20, 2026
**Week:** 7 of 10 — follows Week 6 (Random Forests, closed out)

Random Forest beat its baseline on 8/10 target/direction combos last week, but lost consistently on `class_target_2` (both directions) and `class_target_3_corrected` (LONG) — every tuning stage, not a fluke. This week swaps bagging for boosting: instead of averaging many independent trees to cut variance, XGBoost builds trees sequentially, each one correcting the previous ensemble's residual error, which cuts bias instead. That's a genuinely different failure mode than RF's, which is exactly why it gets a real shot at the two targets RF couldn't close. You'll also re-open the `log1p` question Week 6 skipped without testing, and move from manual grid tuning to Optuna's Bayesian search now that XGBoost has more interacting hyperparameters than a hand grid can reasonably cover.

---

## PART A — Setup (carried forward from Week 6, complete)

### A.1 — Install packages

In [ ]:
!pip install -q supabase scikit-learn statsmodels xgboost shap optuna


### A.2 — Environment detection + secrets (Kaggle-primary, Colab-compatible)

In [ ]:
import os

IS_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ
IS_COLAB = 'COLAB_RELEASE_TAG' in os.environ

if IS_KAGGLE:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    get_secret = lambda k: secrets.get_secret(k).strip()
elif IS_COLAB:
    from google.colab import userdata
    get_secret = lambda k: userdata.get(k)
else:
    get_secret = lambda k: os.environ[k]

print(f"Environment: {'Kaggle' if IS_KAGGLE else 'Colab' if IS_COLAB else 'Other/local'}")


### A.3 — Connect to both databases

In [ ]:
import pandas as pd
import numpy as np
from supabase import create_client

# Main DB
main_client = create_client(get_secret('SUPABASE_URL'), get_secret('SUPABASE_KEY'))

# Analytics DB
analytics_client = create_client(get_secret('ANALYTICS_SUPABASE_URL'), get_secret('ANALYTICS_SUPABASE_KEY'))

print("Connected to both Supabase projects.")


### A.4 — Fetch + merge (keep exactly one merged dataframe)

In [ ]:
df_main = pd.DataFrame(main_client.table('signals').select('*').eq('status', 'analyzed').execute().data)
df_analytics = pd.DataFrame(analytics_client.table('crossover_analytics').select('*').execute().data)

df_analytics_renamed = df_analytics.rename(columns={'crossover_utc': 'checked_at_utc'})
df = df_main.merge(df_analytics_renamed, on=['checked_at_utc', 'symbol'], how='inner')

print(f"Merged shape: {df.shape}")


### A.5 — Targets: classification + regression (carried forward, unchanged from Week 6)
Retired: the old T1–T4 / `target_special` system and `class_target_3`. Do not regenerate either — see the session prompt if an old notebook resurfaces them.

In [ ]:
def optimal_entry_candle(row):
    """Returns 15-min candles from crossover to optimal entry. 0 = immediate entry."""
    try:
        if pd.isnull(row['optimal_entry_utc']) or pd.isnull(row['checked_at_utc']):
            return 0.0
        return float((row['optimal_entry_utc'] - row['checked_at_utc']).total_seconds() / 900)
    except Exception:
        return 0.0

df['Optimum_entry'] = df.apply(optimal_entry_candle, axis=1)

# --- Classification targets ---
mae_abs = df['mae_percent'].abs()
mfe_abs = df['mfe_percent'].abs()

df['target_b']       = (mfe_abs / (mfe_abs + mae_abs + 1e-8)) * (mfe_abs - mae_abs)
df['class_target_1'] = (df['target_b'] > 0.4).astype(int)
df['class_target_2'] = (df['mfe_percent'] > 1).astype(int)

is_long = df['signal_x'].astype(str).str.upper() == 'LONG'
df['total_mae'] = np.where(is_long, df['max_move_down_pct'].astype(float), df['max_move_up_pct'].astype(float))
df['class_target_3_corrected'] = (df['total_mae'] > 1.0).astype(int)
df['class_target_bad'] = ((df['total_mae'] > 1.0) & (df['pnl_percent'] < 0.1)).astype(int)

for t in ['class_target_1', 'class_target_2', 'class_target_3_corrected', 'class_target_bad']:
    print(f"{t}: LONG pos rate = {df.loc[is_long, t].mean():.3f} | SHORT pos rate = {df.loc[~is_long, t].mean():.3f}")

# --- Regression targets (raw columns; log1p applied inside cv_regression per REG_TARGETS bool) ---
df['target_profit']  = df['mfe_percent']
df['target_quality'] = df['target_b']
df['target_danger']  = df['total_mae']
df['target_entry']   = df['Optimum_entry']
df['target_exit']    = df['trade_time'] if 'trade_time' in df.columns else np.nan

REG_TARGETS = {
    'profit':       ('target_profit', True),
    'quality':      ('target_quality', False),
    'danger':       ('target_danger', True),
    'entry_timing': ('target_entry',  True),
    'exit_timing':  ('target_exit',   True),
}


### A.6 — Feature engineering pipeline (unchanged, leak-free, `FE_` prefix)
Analytics outcome columns (`mfe_percent`, `mae_percent`, `pnl_percent`, `trade_duration`, `exit_price`) are post-signal — only valid as lag-1 features. `WRAPPER_FIRST_FINAL_LONG`/`SHORT` still contains the dead reference `'FE_mfe_mae_ratio_lag'` — filtered defensively below, unresolved at the source.

In [ ]:
def safe_ratio(num, den):
    return num / den.replace(0, np.nan)

# --- lag-1 features (safe: previous trade's outcome, not current) ---
df['FE_prev_mfe'] = df.groupby('symbol')['mfe_percent'].shift(1)
df['FE_prev_quality_score'] = df.groupby('symbol')['target_b'].shift(1)

# ... (remaining carried-forward FE_ feature construction from Weeks 3-6 goes here,
#      unchanged — omitted here to keep this section from re-narrating verified work)

FEATURES_ALL = [f for f in [
    # populate from Week 6's finalized feature list
] if f in df.columns]


### A.7 — Feature sets

In [ ]:
feat_set_long = [f for f in FEATURES_ALL if f in df.columns]
feat_set_short = [f for f in FEATURES_ALL if f in df.columns]

df_long = df[df['signal_x'].astype(str).str.upper() == 'LONG'].sort_values('checked_at_utc')
df_short = df[df['signal_x'].astype(str).str.upper() == 'SHORT'].sort_values('checked_at_utc')

print(f"LONG: {df_long.shape}, {len(feat_set_long)} features")
print(f"SHORT: {df_short.shape}, {len(feat_set_short)} features")


### A.8 — Walk-forward CV framework (finalized — `cv_no_scaling`/`cv_regression` are what XGBoost uses, same as RF)
Tree-based models are scale-invariant, so `cv_no_scaling()` applies to `XGBClassifier`/`XGBRegressor` exactly as it did to RF. `cv_regression()`'s `'rmse'` key is genuinely `sqrt(MSE)` — fixed at the source in Week 6, do not reintroduce the unrooted version.

In [ ]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    mean_squared_error, mean_absolute_error, r2_score
)
from sklearn.base import clone

def cv_with_scaling(model, df_subset, feature_cols, target_col, n_splits=5, gap=0):
    df_clean = df_subset[feature_cols + [target_col]].dropna()
    df_clean = df_clean.sort_values('checked_at_utc') if 'checked_at_utc' in df_clean.columns else df_clean
    X = df_clean[feature_cols].values
    y = df_clean[target_col].values
    tscv = TimeSeriesSplit(n_splits=n_splits, gap=gap)
    scores = {'accuracy': [], 'precision': [], 'recall': [], 'f1': [], 'auc': []}
    for train_idx, test_idx in tscv.split(X):
        X_tr, X_te = X[train_idx], X[test_idx]
        y_tr, y_te = y[train_idx], y[test_idx]
        if len(np.unique(y_tr)) < 2 or len(np.unique(y_te)) < 2:
            continue
        scaler = StandardScaler()
        X_tr_sc = scaler.fit_transform(X_tr)
        X_te_sc = scaler.transform(X_te)
        m = clone(model)
        m.fit(X_tr_sc, y_tr)
        y_pred = m.predict(X_te_sc)
        y_prob = m.predict_proba(X_te_sc)[:, 1]
        scores['accuracy'].append(accuracy_score(y_te, y_pred))
        scores['precision'].append(precision_score(y_te, y_pred, zero_division=0))
        scores['recall'].append(recall_score(y_te, y_pred, zero_division=0))
        scores['f1'].append(f1_score(y_te, y_pred, zero_division=0))
        scores['auc'].append(roc_auc_score(y_te, y_prob))
    return {k: (np.mean(v), np.std(v)) for k, v in scores.items()}


def cv_no_scaling(model, df_subset, feature_cols, target_col, n_splits=5, gap=0):
    df_clean = df_subset[feature_cols + [target_col]].replace([np.inf, -np.inf], np.nan).dropna()
    df_clean = df_clean.sort_values('checked_at_utc') if 'checked_at_utc' in df_clean.columns else df_clean
    X = df_clean[feature_cols].values
    y = df_clean[target_col].values
    tscv = TimeSeriesSplit(n_splits=n_splits, gap=gap)
    scores = {'accuracy': [], 'precision': [], 'recall': [], 'f1': [], 'auc': []}
    for train_idx, test_idx in tscv.split(X):
        X_tr, X_te = X[train_idx], X[test_idx]
        y_tr, y_te = y[train_idx], y[test_idx]
        if len(np.unique(y_tr)) < 2 or len(np.unique(y_te)) < 2:
            continue
        m = clone(model)
        m.fit(X_tr, y_tr)
        y_pred = m.predict(X_te)
        y_prob = m.predict_proba(X_te)[:, 1]
        scores['accuracy'].append(accuracy_score(y_te, y_pred))
        scores['precision'].append(precision_score(y_te, y_pred, zero_division=0))
        scores['recall'].append(recall_score(y_te, y_pred, zero_division=0))
        scores['f1'].append(f1_score(y_te, y_pred, zero_division=0))
        scores['auc'].append(roc_auc_score(y_te, y_prob))
    return {k: (np.mean(v), np.std(v)) for k, v in scores.items()}


def cv_regression(model, df_subset, feature_cols, target_info, n_splits=5, gap=0, scale=False, clip_bounds=None):
    """'rmse' is sqrt(MSE) -- fixed at the source, Week 6."""
    target_col, is_log_target = target_info
    df_c = df_subset[feature_cols + [target_col]].replace([np.inf, -np.inf], np.nan).dropna().copy()
    X, y = df_c[feature_cols].values, df_c[target_col].values
    tscv = TimeSeriesSplit(n_splits=n_splits, gap=gap)
    scores = {'mse': [], 'rmse': [], 'mae': [], 'r2': []}
    for tr, te in tscv.split(X):
        Xtr, Xte = X[tr], X[te]
        ytr, yte = y[tr], y[te]
        if scale:
            sc = StandardScaler()
            Xtr = sc.fit_transform(Xtr)
            Xte = sc.transform(Xte)
        m = clone(model)
        if is_log_target:
            ytr_fit = np.log1p(ytr)
            m.fit(Xtr, ytr_fit)
            preds_log = m.predict(Xte)
            train_log_preds = m.predict(Xtr)
            duan_cf = np.mean(np.exp(ytr_fit - train_log_preds))
            preds_raw = (np.exp(preds_log) * duan_cf) - 1
            yte_raw = yte
            if clip_bounds is not None:
                preds_raw = np.clip(preds_raw, clip_bounds[0], clip_bounds[1])
        else:
            m.fit(Xtr, ytr)
            preds_raw = m.predict(Xte)
            yte_raw = yte
        mse = mean_squared_error(yte_raw, preds_raw)
        scores['mse'].append(mse)
        scores['rmse'].append(np.sqrt(mse))
        scores['mae'].append(mean_absolute_error(yte_raw, preds_raw))
        scores['r2'].append(r2_score(yte_raw, preds_raw))
    return {k: (np.mean(v), np.std(v)) for k, v in scores.items()}


### A.9 — Persistent results/model saving
Reconstructed here from the documented rules in the session prompt (immediate per-target save, results folder inside the git repo, pull-merge-push, no force-push, no auto-deleted lock files) — **the actual current `save_progress.py` wasn't in this session's context, so swap this for your real version rather than trusting this as authoritative.**

In [ ]:
import json as _json
import subprocess as _sp

RESULTS_DIR = "EMA-optimizer-pipeline-v2/results"  # must live inside the repo path
os.makedirs(RESULTS_DIR, exist_ok=True)

def save_result_dict(d, name):
    path = os.path.join(RESULTS_DIR, f"{name}.json")
    with open(path, "w") as f:
        _json.dump(d, f, indent=2, default=str)
    return path

def save_model(model, name):
    import joblib
    path = os.path.join(RESULTS_DIR, f"{name}.joblib")
    joblib.dump(model, path)
    return path

def save_progress(commit_message):
    # NOTE (reconstructed, not verified): pull --no-rebase before push, never force-push,
    # never auto-resolve a real conflict or auto-delete .git/index.lock -- surface both as errors.
    raise NotImplementedError("Placeholder -- replace with your actual save_progress.py")


### A.10 — The number to beat: Week 6's `week7_baselines_to_beat`

In [ ]:
with open(os.path.join(RESULTS_DIR, "week7_baselines_to_beat.json")) as f:
    week7_baselines_to_beat = _json.load(f)

print(f"Loaded {len(week7_baselines_to_beat)} target/direction baselines from Week 6.")
print("Reminder: class_target_2 (both directions) and class_target_3_corrected (LONG) are")
print("currently won by Week 3/4 LogReg, not RF -- these are XGBoost's specific targets to beat.")


---

## PART B — Boosting Theory (Monday)

**Concept:** AdaBoost reweights misclassified samples so the next tree focuses on them. Gradient Boosting generalizes this: each new tree is fit to the *residual error* (technically the negative gradient of the loss) of the ensemble so far, not to reweighted samples directly. Learning rate (shrinkage) scales each new tree's contribution down, so the ensemble takes many small corrective steps instead of a few large ones — smaller steps generalize better but need more trees.

**Bagging vs Boosting — think about which reduces which:** RF reduces *variance* (many independent, decorrelated trees averaged together). Boosting reduces *bias* (each tree specifically targets what the ensemble still gets wrong). This isn't just terminology — it's the reason XGBoost gets a real shot at `class_target_2` and `class_target_3_corrected`: if RF's ceiling on those targets is a bias problem (the model class can't represent the relationship well, no matter how many independent trees you average), more bagging was never going to fix it. Boosting might, because it attacks bias directly.

### Hand calculation: 2-tree gradient boosting example (fill in)

In [ ]:
# TODO: pick 4-5 toy data points (can be synthetic, doesn't need real data)
# TODO: fit tree 1 to y directly, record predictions
# TODO: compute residuals = y - tree_1_predictions
# TODO: fit tree 2 to the residuals
# TODO: final_prediction = tree_1_pred + learning_rate * tree_2_pred
# TODO: show this converges toward y as you (mentally) add more trees


### Deliverable — Boosting vs Bagging comparison notes (fill in)
Write with specific reference to your Week 6 targets: which looked bias-limited (RF plateaued regardless of tuning) vs variance-limited (RF's tuning clearly helped)?

*(your notes here)*

---

## PART C — XGBoost Mathematics (Tuesday)

**Concept — Newton boosting:** unlike plain gradient boosting (first-order: fits residuals), XGBoost uses a second-order approximation of the loss — each tree's leaf weight is computed from the *gradient and Hessian* sums of the samples that land in it, not a raw arithmetic mean the way a bagged tree's leaf prediction is. This is the concrete answer to whether the Week 6 `log1p`-skip reasoning ("tree splits are invariant to monotonic transforms") still applies here — trace through the leaf-weight formula yourself below before assuming it does or doesn't.

**Regularization:** `lambda` (L2 on leaf weights) and `alpha` (L1) are XGBoost's mechanism for controlling overfitting — the same goal Week 6 pursued via `max_depth`/`min_samples_leaf`, different mechanism (penalizing leaf weight magnitude directly, rather than restricting tree structure).

**Sparsity-aware split finding:** XGBoost handles missing values natively by learning a default split direction for them, rather than requiring imputation.

### Deliverable — derive the leaf weight formula (fill in)
Work through (or trace from a reference) how a leaf's optimal weight is computed from the gradient/Hessian sums of the samples in it.

*(your derivation/notes here)*

### Your answer to the log1p question — BEFORE testing it Wednesday (fill in)
Given what you just derived about how leaf weights are computed: do you expect skipping `log1p` to matter more, less, or about the same for XGBoost regression targets compared to Week 6's RF? Reasoning first, test comes Wednesday.

*(your reasoned prediction here — not a guess)*

---

## PART D — XGBoost Implementation (Wednesday)

### Train XGBClassifier / XGBRegressor with early stopping
Early stopping (on a genuine walk-forward *validation* fold, not the test fold) replaces Week 6's fixed `n_estimators` search — it tunes the number of trees per-target during training instead of via a separate sweep.

In [ ]:
# ============================================================
# TODO: Train XGBClassifier with early stopping, per classification target
# ============================================================
# TODO: for each of class_target_1/2/3_corrected/bad, both df_long and df_short:
#   TODO: split off a walk-forward validation fold distinct from the final test fold
#   TODO: model = XGBClassifier(n_estimators=<ceiling, not a target>, early_stopping_rounds=..., ...)
#   TODO: model.fit(X_train, y_train, eval_set=[(X_val, y_val)])
#   TODO: evaluate via cv_no_scaling() for comparability with Week 6's RF numbers
xgb_clf_results = {}


In [ ]:
# ============================================================
# TODO: Train XGBRegressor with early stopping, per REG_TARGETS
# ============================================================
# TODO: mirror the classifier loop above for all five REG_TARGETS, both directions
xgb_reg_results = {}


### Handling class imbalance — `scale_pos_weight`
Your most imbalanced targets (`class_target_3_corrected`, `class_target_bad`) are the ones to watch here.

In [ ]:
# TODO: compute scale_pos_weight = (# negative) / (# positive) per target/direction
# TODO: compare CV results with vs without scale_pos_weight for class_target_3_corrected and class_target_bad specifically


### The log1p A/B test — actually run it this time
Pick one regression target. Hold `max_depth`/`learning_rate`/everything else fixed. Run `cv_regression()` with `apply_log1p=True` and again with `False`. Compare the real R² difference — don't inherit Week 6's RF decision without checking.

In [ ]:
# TODO: pick one target (suggestion: 'profit' or 'danger' -- both right-skewed, non-trivial R2 last week)
# TODO: run cv_regression(XGBRegressor(...), df_subset, features, (col, True))
# TODO: run cv_regression(XGBRegressor(...), df_subset, features, (col, False))  -- same hyperparameters
# TODO: print both R2/RMSE side by side -- does this match your Tuesday prediction?


### Compare against Week 6's `week7_baselines_to_beat` — separate tables for classification and regression

In [ ]:
# TODO: classification_comparison = pd.DataFrame([...])  -- AUC only, XGBoost vs week7_baselines_to_beat
# TODO: regression_comparison = pd.DataFrame([...])        -- R2 only, XGBoost vs week7_baselines_to_beat
# TODO: do NOT merge these into one table -- AUC and R2 aren't the same kind of number
# TODO: which targets does XGBoost win? Does it close class_target_2 / class_target_3_corrected (LONG)?


In [ ]:
save_progress("Week 7: XGBoost classifiers + regressors trained with early stopping, log1p A/B tested")


---

## PART E — Hyperparameter Tuning with Optuna (Thursday)

**Concept:** Bayesian optimization models which regions of hyperparameter space are promising based on trials already run, rather than exhaustively covering a fixed grid. With `learning_rate`, `max_depth`, `min_child_weight`, `subsample`, `colsample_bytree`, `lambda`, `alpha` all potentially interacting, a manual grid the way Week 6 staged `max_features` → `n_estimators` → depth/leaf would need far more stages to cover the same space reasonably.

**Time-budget this before launching anything** — Week 6's sweeps ran 2–3x longer than estimated more than once. Time-test one trial first.

In [ ]:
# TODO: time-test ONE Optuna trial (one target/direction) before committing to a study size
# TODO: import time; start = time.time(); <run one trial's worth of training>; elapsed = time.time() - start
# TODO: multiply elapsed by planned n_trials x n_targets x n_directions -- is this actually feasible in one session?


In [ ]:
# ============================================================
# TODO: Define an Optuna objective function using walk-forward CV (cv_no_scaling / cv_regression)
# ============================================================
# TODO: def objective(trial):
#           params = {
#               'learning_rate': trial.suggest_float(...),
#               'max_depth': trial.suggest_int(...),
#               'min_child_weight': trial.suggest_...(...),
#               'subsample': trial.suggest_float(...),
#               'colsample_bytree': trial.suggest_float(...),
#               'lambda': trial.suggest_float(...),
#               'alpha': trial.suggest_float(...),
#           }
#           TODO: return the CV metric to optimize (AUC or R2 depending on target type)
# TODO: study = optuna.create_study(direction='maximize')
# TODO: study.optimize(objective, n_trials=<time-tested count>)
# TODO: save_result_dict(study.best_params, name) immediately per target as each study finishes


### Best trial — does it make sense given Week 6? (fill in)
Does Optuna's best config also converge toward shallow depth + some regularization, matching RF's finding that unbounded depth overfits on this dataset?

*(your notes here)*

---

## PART F — SHAP Values + Week 7 Consolidation (Friday)

### SHAP summary plot — compare against Week 6's permutation importance top-10
Do RF's permutation importance and XGBoost's SHAP values agree more than MDI vs permutation did in Week 6?

In [ ]:
# TODO: import shap
# TODO: explainer = shap.TreeExplainer(xgb_model)
# TODO: shap_values = explainer.shap_values(X_test)
# TODO: shap.summary_plot(shap_values, X_test, feature_names=features)
# TODO: compare top-10 SHAP features against Week 6's permutation importance top-10 for the same target


### SHAP dependence plot — compare shape against Week 6's PDP curve

In [ ]:
# TODO: pick your Week 6 top feature (rsi_4h or ema_separation_4h)
# TODO: shap.dependence_plot(feature_name, shap_values, X_test)
# TODO: does the shape match the PDP curve you built in Week 6 for the same feature? Sanity check, not expected to be identical.


### SHAP waterfall plot — one good signal, one bad signal

In [ ]:
# TODO: pick one row where class_target_1 == 1 and prediction was confident/correct
# TODO: pick one row where class_target_1 == 0 and prediction was confident/correct
# TODO: shap.waterfall_plot(...) for each -- what pushed the prediction each way?


### Week 7 Honest Record
**Write this last — after re-running the cells it summarizes, not from memory of what you meant to get.** Every "..." below gets filled in only once the corresponding cell above has actually been run in this session; leaving one unfilled and moving on is worse than leaving it visibly marked TODO.

In [ ]:
week7_record = f"""
=================================================================================
WEEK 7 PRODUCTION LOG -- GRADIENT BOOSTING I (XGBOOST, CLASSIFICATION + REGRESSION)
=================================================================================

CLASSIFICATION LEADERBOARD (best AUC per target, model + delta over week7_baselines_to_beat)
... (fill in per target, from your actual comparison table above)

REGRESSION LEADERBOARD (best R2 per target, model + delta over week7_baselines_to_beat)
... (fill in per target, from your actual comparison table above)

LOG1P A/B TEST
Result: ...
Does this match your Tuesday prediction?: ...

SHAP vs PERMUTATION IMPORTANCE
Did they agree more than MDI vs permutation did in Week 6?: ...

HONEST VERDICT
Is XGBoost now your best model overall, in each family?: ...
Did it close the class_target_2 / class_target_3_corrected (LONG) gap RF couldn't?: ...
Any target where XGBoost lost to RF or a simpler model, and your best guess why: ...
Biggest surprise this week: ...
Expectation for Week 8 (LightGBM + Stacking): ...
"""

print(week7_record)
# Fill in every "..." only after the cells above have actually run this session.


In [ ]:
save_result_dict(week7_baselines_to_beat, "week8_baselines_to_beat")  # TODO: rename/rebuild once XGBoost leaderboard above is final
save_progress("Week 7 close-out: XGBoost + SHAP, final baselines to beat for Week 8 LightGBM")
